# Modélisation : effet des médailles olympiques sur les licenciés

## Objectif
On cherche à mesurer si les médailles obtenues aux JO (par sport) sont associées à une augmentation du nombre de licenciés les années suivantes.

On utilise deux approches complémentaires :

1) **Modèle économétrique (principal)** : estimation de l’effet moyen des médailles sur la **croissance** des licenciés, avec **effets fixes sport** et **effets fixes année**, et erreurs robustes **clusterisées par sport**.

2) **Diagnostics complémentaires (“preuves”)** : modèles prédictifs simples pour quantifier le rôle de l’inertie (niveau passé) vs les médailles.
Attention: Ces diagnostics sont descriptifs/prédictifs, pas causaux.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Package maison
from model.feature_engineeringv2 import build_model_dataset
from model.modele_econo import fit_modele_croissance
from model.preuvesv2 import eval_ablation_models, ridge_coefficients, test_medals_incremental


## Construction du panel final sport–année

Nous construisons un panel au niveau *(sport, année)* contenant :
- `licences_annuelles` : nombre total de licenciés du sport l’année donnée ;
- des variables descriptives agrégées (sexe, âge, dispersion géographique) ;
- `jo_ref` : JO de référence (dernier JO avant l’année) ;
- `total_medailles` : nombre de médailles obtenues lors du JO de référence pour ce sport (variable clé).


expliquer un peu les feautures que j'ai choisi de créer et le choix de structure de mon dataset + détailler un peu plus + renomer certains features pour plus de clarté + bien précisé qu'ici licences annuelles ce n'est plus les licences annuelles par catégorie d'age mais bien la somme de toutes les licences annuelles i.e. toute les liceces qu'il y a eu dans un sport pour une année)

In [2]:
df = pd.read_parquet("data/data_clean/data_complet.parquet")
# df_lic = agreg_licencies_par_sport_annee(df)
# print(df_lic.columns)
# df_features = build_lic_features(df_lic)
# df_model_full = merge_medals(df_features, df_med)

# print("df_model_full shape:", df_model_full.shape)
df_model_full =  build_model_dataset(df)
df_model_full.head()


,code_sport,annee,licences_annuelles,jo_ref,or,argent,bronze,total_medailles,nb_femmes,nb_hommes,part_femmes,age_mean,age_std,nb_lt14,nb_14_17,nb_18_34,nb_35_49,nb_50p,part_lt14,part_14_17,part_18_34,part_35_49,part_50p,licences_annuelles_lag1,licences_annuelles_lag2,croissance_lag1,croissance_lag2,licences_annuelles_roll2
0,ATH,2016,301976,2016,0.0,3.0,3.0,6.0,141983,0,1.0,28.460622,19.342358,105365.0,36104.0,40191.0,63852.0,56460.0,0.348918,0.119559,0.133093,0.211447,0.186969,NaN,NaN,NaN,NaN,NaN
1,ATH,2017,307018,2016,0.0,3.0,3.0,6.0,146532,0,1.0,28.641338,19.515100,106684.0,36941.0,40206.0,64129.0,59056.0,0.347485,0.120322,0.130956,0.208877,0.192354,301976.0,NaN,0.016697,NaN,NaN
2,ATH,2018,314692,2016,0.0,3.0,3.0,6.0,147945,0,1.0,28.724186,19.756976,110883.0,36432.0,40194.0,63913.0,62268.0,0.352354,0.115770,0.127725,0.203097,0.197870,307018.0,301976.0,0.024995,0.016697,304497.0
3,ATH,2019,316749,2016,0.0,3.0,3.0,6.0,150440,0,1.0,29.010014,19.952825,111257.0,35764.0,40873.0,64225.0,64622.0,0.351247,0.112910,0.129039,0.202763,0.204016,314692.0,307018.0,0.006537,0.024995,310855.0
4,ATH,2020,305914,2016,0.0,3.0,3.0,6.0,144038,0,1.0,29.511235,20.206277,106195.0,33327.0,38872.0,61927.0,65593.0,0.347140,0.108942,0.127068,0.202433,0.214416,316749.0,314692.0,-0.034207,0.006537,315720.5


## Sanity checks (cohérence et reproductibilité)

On vérifie :
- unicité des observations (sport–année) ;
- cohérence de l’assignation `jo_ref` ;
- cohérence de `total_medailles` (constante par sport pour un JO donné).


In [3]:
print("Duplicats (code_sport, annee):", df_model_full.duplicated(["code_sport","annee"]).sum())

# total_medailles doit être constant au sein de (sport, jo_ref)
tmp = df_model_full.groupby(["code_sport","jo_ref"])["total_medailles"].nunique().reset_index()
print("Max nunique(total_medailles) par (sport, jo_ref):", tmp["total_medailles"].max())

# répartition jo_ref par année
df_model_full.groupby(["annee","jo_ref"]).size().head(20)


Duplicats (code_sport, annee): 0
Max nunique(total_medailles) par (sport, jo_ref): 1


annee  jo_ref
2016   2016      34
2017   2016      34
2018   2016      34
2019   2016      34
2020   2016      34
2021   2020      34
2022   2020      34
2023   2020      34
2024   2024      34
dtype: int64

## Modèle économétrique principal : croissance post-JO (panel avec effets fixes)

Nous estimons :

\[
\Delta \log(1+Lic_{s,t}) = \beta \cdot Med_{s,JO} + \alpha_s + \gamma_t + \varepsilon_{s,t}
\]

- Variable dépendante : \(\Delta \log(1+Lic_{s,t})\) ≈ taux de croissance annuel du nombre de licenciés (robuste aux niveaux très différents entre sports).
- \(\alpha_s\) : **effets fixes sport** (popularité structurelle, culture de pratique, taille de base).
- \(\gamma_t\) : **effets fixes année** (chocs communs : Covid, politiques publiques, tendances globales).
- Erreurs standards **clusterisées par sport** : autorise la corrélation intra-sport dans le temps.

Interprétation : si \(\beta\) est petit, \(100\beta\) ≈ variation en points de % de la croissance annuelle associée à **+1 médaille**.


In [3]:
results_econo = fit_modele_croissance(
    df_model_full=df_model_full,
    medals_col="total_medailles",
    start_year=2017,
    end_year=2024,
    controls=["part_femmes", "age_mean", "nb_departements_actifs"],  
)

print(results_econo.summary())


                            OLS Regression Results                            
Dep. Variable:               dlog_lic   R-squared:                       0.568
Model:                            OLS   Adj. R-squared:                  0.489
Method:                 Least Squares   F-statistic:                     11.69
Date:              sam., 27 déc. 2025   Prob (F-statistic):           5.48e-08
Time:                        18:49:11   Log-Likelihood:                 247.20
No. Observations:                 272   AIC:                            -408.4
Df Residuals:                     229   BIC:                            -253.4
Df Model:                          42                                         
Covariance Type:              cluster                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                0.4894 

C:\Users\mcmoi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 43, but rank is 9
  warnings.warn('covariance of constraints does not have full '


In [4]:
beta = float(results_econo.params["med_last"])
ci = results_econo.conf_int().loc["med_last"]

print(f"beta(médailles) = {beta:.6f}")
print(f"Interprétation: +1 médaille -> ~ {100*beta:.2f}% de croissance annuelle (approx.)")
print(f"IC 95%: [{100*ci[0]:.2f}%, {100*ci[1]:.2f}%]")


beta(médailles) = 0.002715
Interprétation: +1 médaille -> ~ 0.27% de croissance annuelle (approx.)
IC 95%: [-0.92%, 1.46%]


### Discussion économétrique (limites)

Même avec effets fixes sport et année, l’interprétation causale doit rester prudente :
- les médailles peuvent être endogènes (sports structurellement forts / mieux financés → plus de médailles et plus de licenciés) ;
- effets hétérogènes selon les sports (médiatisation, accessibilité de la pratique, structure fédérale) ;
- le modèle mesure un effet moyen à court terme, pas une trajectoire complète.

Ces limites motivent les diagnostics ci-dessous, qui mettent en évidence le rôle central de l’inertie dans les licenciés.


## Diagnostics / “preuves” complémentaires

Ces diagnostics ne visent pas à établir une causalité stricte, mais à :
1) quantifier le rôle de l’inertie vs médailles en **prédiction hors échantillon** ;
2) illustrer que la dynamique des licenciés est très persistante ;
3) vérifier si les médailles apportent une information marginale dans le cadre du modèle économétrique.


### Preuve 1 — Ablation prédictive (inertie vs médailles)

On compare trois modèles prédictifs simples sur la période test (2022–2024) :
- **M0 : inertie seule** (`log(lic_{t-1})`)
- **M1 : médailles seules**
- **M2 : inertie + médailles**

Attention: Un bon score des “médailles seules” peut refléter une corrélation avec la popularité structurelle (pas causal).
L’enjeu principal est de comparer M0 vs M2 : l’ajout des médailles apporte-t-il un gain prédictif notable ?


In [8]:
# from sklearn.linear_model import Ridge
# from sklearn.metrics import r2_score, mean_absolute_error

# def eval_ablation_models(df_model_full, train_years=(2017, 2020), test_years=(2022, 2024),
#                          medals_col="total_medailles", alpha=1.0):
#     df = df_model_full.sort_values(["code_sport","annee"]).copy()
#     df["y"] = np.log1p(df["nb_licencies"])
#     df["log_lag1"] = df.groupby("code_sport")["y"].shift(1)
#     df["trend"] = df["annee"] - df["annee"].min()
#     df["med_last"] = df[medals_col]
#     df = df.dropna(subset=["log_lag1"]).copy()

#     train_df = df[df["annee"].between(*train_years)].copy()
#     test_df  = df[df["annee"].between(*test_years)].copy()

#     def fit_predict(feats):
#         Xtr = pd.get_dummies(train_df[["code_sport"] + feats], columns=["code_sport"], drop_first=True).astype("float32")
#         Xte = pd.get_dummies(test_df[["code_sport"] + feats], columns=["code_sport"], drop_first=True).astype("float32")
#         Xte = Xte.reindex(columns=Xtr.columns, fill_value=0).astype("float32")

#         ytr = train_df["y"].astype("float32").values
#         yte = test_df["y"].astype("float32").values

#         m = Ridge(alpha=alpha)
#         m.fit(Xtr, ytr)
#         pred = m.predict(Xte)

#         return float(r2_score(yte, pred)), float(mean_absolute_error(np.expm1(yte), np.expm1(pred)))

#     rows = []
#     for name, feats in [
#         ("M0: inertie seule", ["log_lag1"]),
#         ("M1: médailles seules", ["med_last"]),
#         ("M2: inertie + médailles", ["log_lag1","med_last"]),
#     ]:
#         r2, mae = fit_predict(feats)
#         rows.append({"modele": name, "R2_log": r2, "MAE_niveau": mae})
#     return pd.DataFrame(rows)

ablation = eval_ablation_models(df_model_full)
ablation


,model,R2_log,MAE_niveau
0,M0: inertie seule (lag1),0.989764,49221.171875
1,M1: médailles seules,0.934406,199239.156250
2,M2: inertie + médailles,0.989673,49563.828125


**Lecture** :
- Si M0 est déjà très performant, cela indique que les licenciés sont fortement **inertiels**.
- Si M2 n’améliore que marginalement M0, les médailles apportent peu d’information prédictive additionnelle dans ce cadre.
- La performance de M1 doit être interprétée prudemment (corrélation structurelle possible).


## Commentaire détaillé Preuve 1 - Modèles d’ablation : inertie vs médailles

Afin d’évaluer le rôle explicatif des performances olympiques sur l’évolution du nombre de licenciés par sport, nous mettons en place une série de **modèles prédictifs en ablation**.  
L’objectif n’est pas ici de maximiser la performance prédictive, mais de **comparer différentes spécifications** afin d’identifier l’apport informationnel marginal des médailles, relativement à l’inertie propre des effectifs de licenciés.

### Méthodologie

La variable cible est définie comme le logarithme du nombre de licences annuelles :
\[
y_{s,t} = \log(1 + \text{licences}_{s,t})
\]

Trois spécifications sont estimées à l’aide d’une **régression Ridge**, avec effets fixes sport (encodés par variables indicatrices) :

- **M0 – Inertie seule** : lag(1) de la variable cible  
- **M1 – Médailles seules** : nombre total de médailles obtenues lors des Jeux Olympiques de référence  
- **M2 – Inertie + médailles** : combinaison des deux précédentes

Le découpage temporel est strictement respecté afin d’éviter toute fuite d’information :
- période d’entraînement : 2017–2020
- période de test : 2022–2024

Les performances sont évaluées à l’aide de :
- \(R^2\) sur la cible en log  
- MAE sur la cible en niveau (après transformation inverse)

### Résultats

Les résultats obtenus sont résumés dans le tableau ci-dessus. Ils mettent en évidence plusieurs points clés :

- Le modèle **M0 (inertie seule)** affiche une performance très élevée, avec un \(R^2\) proche de 0.99.  
  Cela indique que le nombre de licenciés est **fortement inerte** : le niveau observé une année est très largement expliqué par celui de l’année précédente.

- Le modèle **M1 (médailles seules)** est nettement moins performant, en particulier en termes d’erreur absolue moyenne sur la cible en niveau.  
  Les médailles, prises isolément, ne permettent donc pas de prédire efficacement le niveau des licences.

- Le modèle **M2 (inertie + médailles)** n’améliore que marginalement les performances du modèle M0.  
  L’ajout des médailles apporte peu d’information prédictive supplémentaire une fois l’inertie prise en compte.

### Interprétation

Ces résultats suggèrent que, sur la période étudiée, **la dynamique des licences sportives est dominée par des mécanismes d’inertie**, tandis que l’effet des performances olympiques est limité à court et moyen terme.

L’association observée entre médailles et licenciés peut refléter des corrélations structurelles (sports historiquement populaires et performants), plutôt qu’un effet causal direct des succès olympiques sur les inscriptions.

### Limites et prolongements

- La performance très élevée du modèle M0 invite à interpréter les gains marginaux avec prudence : dans un contexte fortement inerte, il est mécaniquement difficile pour toute autre variable d’améliorer significativement la prédiction.
- Les métriques en niveau sont dominées par les sports à fort volume de licenciés ; une analyse complémentaire en termes relatifs pourrait affiner la lecture.
- Cette preuve prédictive ne permet pas de conclure à une causalité : une approche économétrique dédiée (effets fixes, variables instrumentales, analyses événementielles) serait nécessaire pour aller plus loin.

Cette première preuve constitue néanmoins un point d’appui solide pour la suite de l’analyse, en mettant en évidence le rôle central de l’inertie et les limites explicatives des médailles dans un cadre prédictif.


### Preuve 2 — Poids de l’inertie (coefficients Ridge)

On estime un Ridge sur le train et on examine les coefficients (en valeur absolue).
Si `log_lag1` domine nettement les autres, cela quantifie le rôle central de l’inertie.


In [5]:
coef = ridge_coefficients(df_model_full)
coef.head(15)


log_lag1          0.980910
code_sport_HAL   -0.089050
code_sport_SUR   -0.085293
code_sport_CAK    0.071467
code_sport_DIV    0.059085
code_sport_FOO    0.049534
code_sport_BAS   -0.048638
code_sport_NAT    0.045447
code_sport_ESD    0.040295
code_sport_HOC   -0.039005
code_sport_GYM    0.038292
code_sport_VOI   -0.037130
code_sport_TIR    0.032394
code_sport_TAE   -0.032342
code_sport_AVI   -0.027619
dtype: float32

**Lecture** :
- un coefficient très élevé sur `log_lag1` reflète l’inertie : le niveau passé explique la majeure partie du niveau actuel.
- les médailles ont généralement un poids plus faible dans la prédiction du niveau.


### Lecture et interprétation des coefficients Ridge

La figure ci-dessus présente les coefficients estimés par un modèle Ridge entraîné
sur la période 2017–2020. Les coefficients sont triés par valeur absolue décroissante,
ce qui permet d’identifier les variables les plus informatives dans un cadre prédictif.

Un premier résultat saillant est la **domination très nette du coefficient associé à
`log_lag1`**, c’est-à-dire au logarithme du nombre de licences de l’année précédente.
Son coefficient est largement supérieur à celui de toutes les autres variables,
ce qui confirme quantitativement le **rôle central de l’inertie** dans la dynamique
des licenciés sportifs.

Les coefficients associés aux variables indicatrices de sport (`code_sport_*`)
présentent des amplitudes beaucoup plus faibles. Ils capturent principalement des
**différences structurelles de niveau entre sports**, mais n’expliquent qu’une part
marginale de la variation conditionnellement à l’inertie. Leur interprétation doit
donc rester descriptive et non causale.

La variable de médailles (`med_last`) et la tendance globale (`trend`) apparaissent
avec des coefficients nettement plus faibles que `log_lag1`. Cela suggère que, dans
un cadre prédictif, **les performances olympiques et la tendance temporelle globale
apportent peu d’information supplémentaire** une fois la dynamique passée des licences
prise en compte.

Ces résultats sont pleinement cohérents avec ceux obtenus dans la *Preuve 1* :
- la très forte performance du modèle basé uniquement sur l’inertie,
- le gain marginal apporté par l’ajout des médailles,
- et la difficulté pour toute autre variable de concurrencer le pouvoir explicatif
  du niveau passé des licenciés.

### Limites et portée de l’analyse

Il est important de souligner que cette analyse repose sur un modèle Ridge à visée
**prédictive**. Les coefficients estimés ne doivent pas être interprétés comme des
effets causaux. En particulier, la faiblesse relative du coefficient des médailles
ne signifie pas l’absence totale d’effet, mais plutôt que cet effet est dominé par
des mécanismes d’inertie et de structure propres aux sports.

Néanmoins, cette seconde preuve apporte un éclairage complémentaire utile : elle
quantifie explicitement le poids relatif des différentes composantes du modèle et
renforce l’idée que la dynamique des licences sportives est avant tout auto-persistante.


### Preuve 3 — Les médailles apportent-elles une information marginale dans le modèle éco ?

On compare :
- un modèle **sans** médailles (FE sport + FE année + contrôles)
- un modèle **avec** médailles

On réalise un test de Wald sur l’hypothèse nulle \(H_0 : \beta = 0\) (coefficient médailles nul).


In [3]:

m0, m1, wald = test_medals_incremental(df_model_full)
print(wald)
print("Coefficient médailles :", m1.params.get("med_last", np.nan))

print("beta medals:", m1.params["med_last"])


<Wald test (chi2): statistic=[[0.2001553]], p-value=0.6545955177395218, df_denom=1>
Coefficient médailles : 0.0027154461889349903
beta medals: 0.0027154461889349903


C:\Users\mcmoi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\base\model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(


**Lecture** :
- Si la p-value est faible, l’effet des médailles est statistiquement différent de zéro, une fois contrôlés les effets fixes.
- Cela renforce l’association conditionnelle mesurée, sans garantir une causalité parfaite.


### Résultats du test économétrique incrémental

La *Preuve 3* vise à évaluer formellement l’apport incrémental des performances
olympiques sur la **croissance annuelle des licences**, en contrôlant pour des
effets fixes sport et année ainsi que pour des variables de contrôle disponibles.

La variable dépendante est définie comme la variation du logarithme du nombre de
licences annuelles :
\[
d\log(\text{licences}_{s,t}) = \log(1 + \text{licences}_{s,t}) - \log(1 + \text{licences}_{s,t-1})
\]

Deux modèles sont estimés :
- **M0** : modèle sans variable de médailles,
- **M1** : modèle incluant la variable de médailles associées aux Jeux Olympiques
  de référence (`med_last`).

L’hypothèse nulle testée est :
\[
H_0 : \beta_{\text{med\_last}} = 0
\]

Le test de Wald conduit aux résultats suivants :
- statistique de test \(\chi^2 \approx 0.20\),
- p-value \(\approx 0.65\).

Cette p-value largement supérieure aux seuils usuels (5 % ou 10 %) conduit à **ne
pas rejeter l’hypothèse nulle**. Autrement dit, **l’effet des médailles sur la
croissance des licences n’est pas statistiquement significatif** une fois pris en
compte les effets fixes sport et année.

Le coefficient estimé associé aux médailles est par ailleurs **très faible en
amplitude** (environ 0.003), ce qui suggère un impact économique limité, même en
l’absence de considération statistique.

### Interprétation et mise en perspective

Ces résultats confirment et renforcent les conclusions tirées des preuves
précédentes :

- La *Preuve 1* montrait que l’ajout des médailles n’améliore que marginalement la
  performance prédictive d’un modèle déjà dominé par l’inertie.
- La *Preuve 2* mettait en évidence le poids écrasant de la variable de lag par
  rapport aux autres signaux.
- La *Preuve 3* apporte une validation économétrique formelle : **les médailles ne
  présentent pas d’effet incrémental significatif sur la croissance des licences**
  dans le cadre spécifié.

L’ensemble de ces résultats suggère que la dynamique des licences sportives est
avant tout gouvernée par des mécanismes d’inertie et de structure propres aux
disciplines, tandis que les performances olympiques jouent, au mieux, un rôle
secondaire et non systématique.


### Portée des résultats et limites

Les résultats obtenus permettent de répondre de manière claire à la question
posée dans le cadre du projet : **aucun effet incrémental significatif des
médailles olympiques sur la croissance des licences n’est mis en évidence**
une fois contrôlée l’inertie et les effets fixes structurels.

Les limites évoquées ci-dessous ne remettent pas en cause cette conclusion,
mais en précisent le périmètre d’interprétation.

Tout d’abord, l’analyse repose sur des données annuelles agrégées au niveau
sport–année. Ce choix est cohérent avec la disponibilité des données, mais
ne permet pas d’identifier des effets de court terme ou des dynamiques
d’anticipation fines autour des événements olympiques.

Ensuite, le cadre retenu privilégie une approche parcimonieuse et robuste,
limitant volontairement la complexité des modèles afin d’éviter des problèmes
de sur-paramétrisation et de faible puissance statistique. Des effets
hétérogènes selon les sports ne sont donc pas explorés ici.

Enfin, l’objectif du projet n’est pas d’établir une relation causale stricte,
mais d’évaluer empiriquement l’existence d’un lien mesurable et robuste dans
un cadre réaliste et reproductible. À ce titre, l’absence d’effet significatif
constitue un résultat informatif en soi.

Dans l’ensemble, les trois preuves mobilisées convergent vers la même
conclusion et apportent une réponse cohérente et étayée à la problématique
initiale.


## Modèle prédictif secondaire (outil de visualisation, pas causal)

On utilise le modèle Ridge de notre package pour produire :
- une performance globale (R² en log et MAE en niveau),
- des courbes observé vs prédit sur la période test, sport par sport.

Ce modèle est surtout utilisé pour illustrer la dynamique (inertie) et visualiser les trajectoires.

## Conclusion (modélisation)

- Le modèle économétrique (panel avec effets fixes sport et année, erreurs cluster sport) fournit une estimation interprétable de l’effet moyen des médailles sur la croissance des licenciés.
- Les diagnostics montrent que la dynamique des licenciés est fortement dominée par l’inertie (niveau passé), et que l’information apportée par les médailles est plus marginale.
- La causalité doit être interprétée avec prudence (endogénéité possible), mais la démarche est cohérente et reproductible.
